# Ground Truth Generation - GridMind (StudyGrid FAQ)

Matches lessons `02-ground-truth.md` and `03-ground-truth-batch.md`, adapted to our project:

- data comes from `../data/studygrid_faq.json` (34 documents, each with a stable `id`)
- no `course` filtering needed - our dataset only has StudyGrid documents
- uses a plain OpenAI client (structured output via `responses.parse`), separate from the
  Groq client the running app uses in `config.py`


## 1. Load the documents

Notebook lives in `evaluation/`, so the data path is one level up.


In [1]:
import sys
sys.path.insert(0, "..")

from ingest import load_faq_data

documents = load_faq_data(path="../data/studygrid_faq.json")
len(documents)


34

In [2]:
documents[0]

{'id': 1,
 'section': 'Getting Started',
 'question': 'What is StudyGrid?',
 'answer': 'StudyGrid is a mobile app that combines group chat, class materials, shared and personal to-do lists, and smart notifications — all in one place for students.'}

## 2. Structured output schema

We want the model to return a Python object, not free text.


In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


## 3. Instructions for the question generator


In [4]:
data_gen_instructions = """
You emulate a StudyGrid user trying to understand how a feature works.
Formulate 5 questions this user might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be
complete and not too short. If possible, use as few words as possible from
the record.

The output should resemble how people ask questions on a support chat.
Not too formal, not too short, not too long.

Only ask about what this specific record actually answers.
Do not introduce features, platforms, or details that are not mentioned in the record.
""".strip()


## 4. OpenAI client

This notebook uses a plain OpenAI client (`OPENAI_API_KEY` in `.env`), separate from the
Groq client the app uses at runtime. Structured output (`responses.parse`) is what we're
relying on here, and it's the officially supported path on OpenAI's API.


In [5]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


## 5. Try it on one document


In [6]:
import json

doc = documents[0]
user_prompt = json.dumps(doc)

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]



In [7]:
response = openai_client.responses.parse(
    model="openai/gpt-oss-120b",
    input=messages,
    text_format=Questions
)

response.output_parsed.questions


['Can you explain what StudyGrid actually is?',
 'What does StudyGrid do for students?',
 'Which features are combined in the StudyGrid app?',
 'Is StudyGrid only a group chat tool or does it have other functions?',
 'How does StudyGrid help manage class materials and to‑do lists?']

## 6. Use the reusable helper

`evaluation_utils.py` sits next to this notebook.


In [8]:
from evaluation_utils import generate_structured

result, usage = generate_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)
print(usage)


['What features does StudyGrid offer for students?', 'Does StudyGrid include a group chat function?', 'Can I use StudyGrid to share class materials with classmates?', 'Does StudyGrid allow me to manage shared and personal to-do lists?', 'What kind of smart notifications does StudyGrid provide?']
ResponseUsage(input_tokens=355, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=614, output_tokens_details=OutputTokensDetails(reasoning_tokens=539), total_tokens=969)


In [9]:
from evaluation_utils import calc_call_cost

calc_call_cost(usage)


{'input_cost': 2.6625e-05,
 'output_cost': 0.00018419999999999998,
 'total_cost': 0.000210825}

## 7. Wrap it: one document -> ground truth records


In [10]:
from evaluation_utils import generate_structured_with_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = generate_structured_with_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []
    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage


In [11]:
generate_ground_truth(doc)

([{'question': 'How do I start a group chat with classmates in StudyGrid?',
   'document': 1},
  {'question': 'Where can I find my class materials within the app?',
   'document': 1},
  {'question': 'Can I share my to-do lists with other students?',
   'document': 1},
  {'question': 'Will StudyGrid send me smart notifications for assignments?',
   'document': 1},
  {'question': 'Is StudyGrid available on mobile devices?', 'document': 1}],
 ResponseUsage(input_tokens=355, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=843, output_tokens_details=OutputTokensDetails(reasoning_tokens=769), total_tokens=1198))

## 8. Sanity check: sequential run on the first 5 documents


In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
len(ground_truth)

25

## 9. Full run, in parallel

34 documents, 6 workers - well under any rate limit concern.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=1) as pool:
    gt_results  = map_progress(pool, documents, generate_ground_truth)


  0%|          | 0/34 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in gt_results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

ValueError: too many values to unpack (expected 2)

## 10. Total cost

Note: `calc_price` uses placeholder per-token prices - check `evaluation_utils.py`.


In [ ]:
from evaluation_utils import calc_total_cost

calc_total_cost(usages)

0.006264675

In [ ]:

calc_call_cost(usage)

{'input_cost': 2.565e-05,
 'output_cost': 0.00016289999999999998,
 'total_cost': 0.00018854999999999998}

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'What does the StudyGrid app do for students?', 'document': 5},
 {'question': 'Is StudyGrid available as a mobile app?', 'document': 5},
 {'question': 'What features can I find in StudyGrid?', 'document': 5},
 {'question': 'Does StudyGrid let me chat with classmates?', 'document': 5},
 {'question': 'Can I manage my class notes and tasks in one place with StudyGrid?',
  'document': 5}]

## 11. Save the ground truth set


In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [ ]:
df_ground_truth.head()

,question,document
0,How can I use the group chat feature on StudyG...,1
1,Where can I access my class materials within t...,1
2,Can I create both shared and personal to-do li...,1
3,What kind of smart notifications does StudyGri...,1
4,Is StudyGrid suitable for all kinds of students?,1


In [ ]:
from ingest import load_faq_data
from retriever import Retriever
from prompts import INSTRUCTIONS, USER_PROMPT_TEMPLATE

documents = load_faq_data(path="../data/studygrid_faq.json")

retriever = Retriever(
    documents,
    instructions=INSTRUCTIONS,
    prompt_template=USER_PROMPT_TEMPLATE
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def vector_search(query):
    return retriever.search(query, num_results=5)

In [ ]:
q = ground_truth[0]
q

{'question': 'How can I use the group chat feature on StudyGrid?',
 'document': 1}

In [ ]:
doc_id = q["document"]
results = vector_search(query=q["question"])

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

1 == 1: True
16 == 1: False
32 == 1: False
17 == 1: False
6 == 1: False


In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
ground_truth_sample = ground_truth[:15]
relevance_total_sample = compute_relevance_total(ground_truth_sample, vector_search)
relevance_total_sample

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [ ]:
for i in [0, 5, 10, 20]:
    q = ground_truth[i]
    r = vector_search(query=q["question"])
    print(i, len(r))

0 5
5 5
10 5
20 5


In [ ]:
import retriever
print(retriever.__file__)

d:\LLM LEARNING\project\studygrid-app-copilot\evaluation\..\retriever.py


In [ ]:
import inspect
print(inspect.getsource(retriever.Retriever.__init__))

    def __init__(
        self,
        documents,
        instructions,
        prompt_template,
        model_name="all-MiniLM-L6-v2",
        db_path="data/faq_vectors.db",
        keyword_fields=("section",),
    ):
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = SentenceTransformer(model_name)

        # If the .db file already exists, the index was built on a
        # previous run — open it as-is, no need to re-embed anything.
        index_already_built = os.path.exists(db_path)

        # NOTE: mode="lsh" (the sqlitesearch default) hashes vectors into
        # ~65k buckets (hash_size=16, n_tables=8). With only a few dozen
        # documents, most buckets end up empty or hold a single vector,
        # so search() can silently return 0-4 results instead of the
        # requested num_results even when a relevant document exists.
        # We use IVF with a single cluster instead: every document lands
        # in th